In [0]:
%run /Configurations/Init_Scripts

In [0]:
dbutils.widgets.text("PromotionUUID", "")
PromotionUUID = dbutils.widgets.get("PromotionUUID")


dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")

dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')

dbutils.widgets.text("ExternalLocationName_silver", "/mntprod_silver")
ExternalLocationName_silver = dbutils.widgets.get("ExternalLocationName_silver")
destination_table_path = ExternalLocation_silver+'/FACT_DifferentShipto_Smartcards'
# destination_table_path =  '/mnt/silver/FACT_DifferentShipto_Smartcards'

PromotionUUID=PromotionUUID.split(",")
print(PromotionUUID)

In [0]:
spark.conf.set("spark.sql.catalog.current", CatalogName)

In [0]:
from pyspark.sql.functions import lit, current_timestamp, desc
Promotion_UUID = ', '.join([f"'{name}'" for name in PromotionUUID])

df = spark.sql(f'''
WITH 
cycles AS (
    SELECT  
        ul.CardSPSerialNbr AS SmartCard,
        ul.ExternalSerialNbr, 
        ul.ShipToAccountID AS ShipTO_of_usage, 
        ul.SoldToAccountID AS SoldTo_of_usage, 
        ul.HdrStartDateGeneratedDt AS Cycle_Date, 
        DP.PromotionName,
        DP.CountryAlpha2Cd,
        count(1) As Cycles
    FROM silverzone.factullogs ul
    INNER JOIN promotion.dim_smartcard AS DS
          ON DS.SmartCardSerialNumber = ul.CardSPSerialNbr
    INNER JOIN promotion.dim_Promotion As DP
          ON DS.PromotionUUID = DP.PromotionUUID
          AND ul.FileNameDeviceTypeCd = DP.DeviceTypeCd      /*condition added to get the promotion elgible devices only*/
    INNER JOIN Silverzone.dimcustomermaster AS sdc 
            ON ul.ShipToAccountID = sdc.AccountID
            AND DP.CountryAlpha2Cd = sdc.AccountCountryAlpha2Cd
    Where HdrStartDateGeneratedDt >= DP.EffectiveDate 
      AND ul.UpdatedDt >= current_timestamp() - INTERVAL 2 DAYS      /*filter added to pass only through last 2 days data*/
    GROUP BY ALL
), 

cardsent AS (
    SELECT 
        eq.ShipToID AS ShipToAccountCardSent,
        eq.ShipStartDt, 
        eq.ShipEndDt, 
        eq.SerialNumberNbr,
        eq.soldtoid AS SoldToAccountIDCardSentto,
        sdc.AccountPrimaryNm AS SoldToAccountNameCardSentto,
        shc.AccountPrimaryNm AS ShipToAccountNameCardSentto,
        shc.AccountCityNm AS ShipToAccountCityNameCardSentto,
        shc.AcctStateProvinceCd AS ShipToAccountStateCdCardSentto,
        shc.AccountCountryAlpha2Cd AS ShipToAccountCountryAlpha2CdCardSentto
    FROM silverzone.dimequipmentmaster AS eq
    LEFT JOIN silverzone.dimcustomermaster AS sdc ON eq.SoldToID = sdc.AccountID
    LEFT JOIN silverzone.dimcustomermaster AS shc ON eq.ShiptoID = shc.AccountID
    WHERE EquipmentTransitStatusNm = 'SHIPPED'
),

DimAccount AS
(
  Select DA.*
  from promotion.dim_account AS DA       /*fetching complete dim_account table*/
),

data AS (
    SELECT DISTINCT
        cycles.SmartCard,
        cycles.ExternalSerialNbr AS ExternalSerialNbr,
        cv.ShipToAccountCardSent , 
        cycles.ShipTo_of_usage as ShipToofusage, 
        cycles.SoldTo_of_usage as SoldToofusage, 
        cycles.Cycle_Date as CycleDate, 
        sdc.AccountPrimaryNm AS SoldToAccountNameofusage,
        shc.AccountPrimaryNm AS ShipToAccountNameofusage,
        shc.AccountCityNm AS ShipToAccountCityNameofusage,
        shc.AcctStateProvinceCd AS ShipToAccountStateCdofusage,
        cv.SoldToAccountIDCardSentto,
        cv.SoldToAccountNameCardSentto,
        cv.ShipToAccountNameCardSentto,
        cv.ShipToAccountCityNameCardSentto,
        cv.ShipToAccountStateCdCardSentto,
        Cycles,
        CASE 
            WHEN a.ShipToAccountId is not null THEN 1 ELSE 0       /*replaced p3 flag eligibility from perpatientfeeflag to shipto accountid check*/
        END AS ShipToofusageP3Flag,
        CASE 
            WHEN b.ShipToAccountId is not null THEN 1 ELSE 0        /*replaced card sent to flag criteria from perpatientfeeflag to shipto accountid check*/
        END AS ShipToAccountCardSenttoFlag,
        cycles.PromotionName AS PromotionName,
        cycles.CountryAlpha2Cd AS Country
    FROM cycles
    LEFT JOIN cardsent cv ON cv.SerialNumberNbr = cycles.SmartCard     
    AND cycles.Cycle_Date >= cv.ShipStartDt
    AND cycles.cycle_date < cv.ShipEndDt
    AND cycles.CountryAlpha2Cd = cv.ShipToAccountCountryAlpha2CdCardSentto
    LEFT JOIN silverzone.dimcustomermaster AS sdc ON cycles.SoldTo_of_usage = sdc.AccountID
    LEFT JOIN silverzone.dimcustomermaster AS shc ON cycles.ShipTO_of_usage = shc.AccountID
    LEFT JOIN DimAccount a ON a.ShipToAccountId = cycles.ShipTO_of_usage  
    AND cycles.cycle_date BETWEEN a.EffectiveDate AND a.TerminationDate
    LEFT JOIN DimAccount b ON b.ShipToAccountId = cv.ShipToAccountCardSent 
    AND cycles.cycle_date BETWEEN b.EffectiveDate AND b.TerminationDate
)     
SELECT * 
FROM data 
WHERE 
    ShipToAccountCardSent != ShipToofusage
    AND ShipToofusageP3Flag = 0
    
''')

df = (df.withColumn("InvestigationMoxieCaseNumber", lit(None))
        .withColumn("InvestigationMoxieCaseStatus", lit(None))
        .withColumn("InvestigationMoxieCaseId", lit(None))
        .withColumn("InvestigationMoxieCaseDate", lit(None))
        .withColumn("PenaltyMoxieCaseNumber", lit(None))
        .withColumn("PenaltyMoxieCaseStatus", lit(None))
        .withColumn("PenaltyMoxieCaseId", lit(None))
        .withColumn("PenaltyMoxieCaseDate", lit(None))
        .withColumn('CreatedBy', lit('ADB_Promotion'))
        .withColumn('CreatedDate', current_timestamp())
        .withColumn('UpdatedBy', lit('ADB_Promotion'))
        .withColumn('UpdatedDate', current_timestamp())                        
   )

display(df.orderBy(desc('CycleDate')))


In [0]:
from delta.tables import DeltaTable

# destination_table_path =  '/mnt/silver/FACT_DifferentShipto_Smartcards'
Deltatbl_EquipmentMaster = DeltaTable.forPath(spark, destination_table_path)
(Deltatbl_EquipmentMaster.alias("tgt")
    .merge(df.alias("src"), """tgt.SmartCard = src.SmartCard 
           and tgt.CycleDate = src.CycleDate
           and tgt.ExternalSerialNbr = src.ExternalSerialNbr
           and tgt.ShipToAccountCardSent = src.ShipToAccountCardSent
           and tgt.ShipToofusage = src.ShipToofusage
           and tgt.SoldToofusage = src.SoldToofusage 
           """)
    .whenNotMatchedInsertAll()
)   .execute()
